In [1]:
!git clone https://github.com/JupaaF/Proyecto_Final_PINNs

Cloning into 'Proyecto_Final_PINNs'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 144 (delta 41), reused 58 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 47.12 MiB | 12.30 MiB/s, done.
Resolving deltas: 100% (41/41), done.


In [2]:
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("wandb_api")
    wandb.login(key=api_key)
    anony = None
except:
    anony = "must"
    print('If you want to use your W&B account, go to Add-ons -> Secrets and provide your W&B access token. Use the Label name as wandb_api. \nGet your W&B access token from here: https://wandb.ai/authorize')

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from scipy.ndimage import distance_transform_edt
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset, random_split
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from scipy.stats.qmc import LatinHypercube, scale # Necesario para tu función
from scipy.spatial import cKDTree

# COSAS DE DATOS

In [4]:
datos_path = 'Proyecto_Final_PINNs/rt.csv'
datos = pd.read_csv(datos_path)

# Rename columns
datos = datos.rename(columns={'Points:0': 'x', 'Points:1': 'y', 'Points:2': 'z',
                              'U:0': 'Ux', 'U:1': 'Uy', 'U:2': 'Uz'})
datos = datos.drop('z', axis=1)
datos = datos.drop('Uz', axis=1)

datos.to_csv(datos_path, index=False)

g = [0, 9.81, 0]
rhoWater = 998
rhoAir = 1.2

muWater = 1.002e-3
muAir = 1.81e-5

# Calculate p for rows where Time is 0 based on alpha.water
datos.loc[(datos['Time'] == 0) & (datos['alpha.water'] > 0.5), 'p'] = datos.loc[(datos['Time'] == 0) & (datos['alpha.water'] > 0.5), 'p_rgh'] + rhoWater * g[1] * datos.loc[(datos['Time'] == 0) & (datos['alpha.water'] > 0.5), 'y']
datos.loc[(datos['Time'] == 0) & (datos['alpha.water'] <= 0.5), 'p'] = datos.loc[(datos['Time'] == 0) & (datos['alpha.water'] <= 0.5), 'p_rgh'] + rhoAir * g[1] * datos.loc[(datos['Time'] == 0) & (datos['alpha.water'] <= 0.5), 'y']

datos.drop('p_rgh', axis=1, inplace=True)
datos.to_csv(datos_path, index=False)

##FORMATEO DE DATOS

L_ref = 0.584

# U_ref: Velocidad característica (magnitud máxima de velocidad en los datos)
vel_mag = np.sqrt(datos['Ux']**2 + datos['Uy']**2)
U_ref = vel_mag.max()

# RHO_ref: Densidad de referencia (usamos la del agua)
RHO_ref = rhoWater

if U_ref < 1e-6:
    print(f"Advertencia: U_ref (max |U| = {U_ref}) es casi cero. Usando 1.0 por defecto.")
    U_ref = 1.0

# --- Escalas derivadas ---
T_ref = L_ref / U_ref      # Tiempo característico
P_ref = RHO_ref * (U_ref ** 2) # Presión característica (presión dinámica)

print("--- Escalas Características Calculadas ---")
print(f"  L_ref (Max y):     {L_ref:.4f} m")
print(f"  U_ref (Max |U|):   {U_ref:.4f} m/s")
print(f"  RHO_ref (Agua):    {RHO_ref:.1f} kg/m^3")
print(f"  T_ref (L/U):       {T_ref:.4f} s")
print(f"  P_ref (rho*U^2):   {P_ref:.2f} Pa")
print("------------------------------------------")

--- Escalas Características Calculadas ---
  L_ref (Max y):     0.5840 m
  U_ref (Max |U|):   6.0033 m/s
  RHO_ref (Agua):    998.0 kg/m^3
  T_ref (L/U):       0.0973 s
  P_ref (rho*U^2):   35967.10 Pa
------------------------------------------


In [5]:
datos = pd.read_csv('/kaggle/working/Proyecto_Final_PINNs/datos_validation_primero.csv')

datos['x'] = datos['x'] / L_ref
datos['y'] = datos['y'] / L_ref
datos['Time'] = datos['Time'] / T_ref
datos['Ux'] = datos['Ux'] / U_ref
datos['Uy'] = datos['Uy'] / U_ref
datos['p'] = datos['p'] / P_ref
datos.to_csv('/kaggle/working/Proyecto_Final_PINNs/datos_validation_primero.csv', index=False)

datos = pd.read_csv('/kaggle/working/Proyecto_Final_PINNs/datos_validation_segundo.csv')

datos['x'] = datos['x'] / L_ref
datos['y'] = datos['y'] / L_ref
datos['Time'] = datos['Time'] / T_ref
datos['Ux'] = datos['Ux'] / U_ref
datos['Uy'] = datos['Uy'] / U_ref
datos['p'] = datos['p'] / P_ref
datos.to_csv('/kaggle/working/Proyecto_Final_PINNs/datos_validation_segundo.csv', index=False)

In [6]:
Re_ref = RHO_ref * U_ref * L_ref / muWater

datos = pd.read_csv('/kaggle/working/Proyecto_Final_PINNs/rt.csv')
# --- Aplicar la adimensionalización ---
datos['x'] = datos['x'] / L_ref
datos['y'] = datos['y'] / L_ref
datos['Time'] = datos['Time'] / T_ref
datos['Ux'] = datos['Ux'] / U_ref
datos['Uy'] = datos['Uy'] / U_ref
datos['p'] = datos['p'] / P_ref

# NOTA: 'alpha.water' ya es adimensional (0 a 1), por lo que NO se toca.

print("Adimensionalización completada.")

## 3. GUARDAR LOS DATOS PROCESADOS

# Guardar los datos procesados en un NUEVO archivo
nondim_datos_path = 'Proyecto_Final_PINNs/rt_nondimensional.csv'
datos.to_csv(nondim_datos_path, index=False)

print(f"Datos adimensionalizados guardados en: {nondim_datos_path}")

Adimensionalización completada.
Datos adimensionalizados guardados en: Proyecto_Final_PINNs/rt_nondimensional.csv


In [7]:
datos_leftWall = pd.read_csv('/kaggle/working/Proyecto_Final_PINNs/leftWall_temporal_puntos.csv')

datos_leftWall['x'] = datos_leftWall['x'] / L_ref
datos_leftWall['y'] = datos_leftWall['y'] / L_ref
datos_leftWall['t'] = datos_leftWall['t'] / T_ref

datos_leftWall.to_csv('/kaggle/working/Proyecto_Final_PINNs/leftWall_temporal_puntos.csv',index = False)

datos_rightWall = pd.read_csv('/kaggle/working/Proyecto_Final_PINNs/rightWall_temporal_puntos.csv')

datos_rightWall['x'] = datos_rightWall['x'] / L_ref
datos_rightWall['y'] = datos_rightWall['y'] / L_ref
datos_rightWall['t'] = datos_rightWall['t'] / T_ref

datos_rightWall.to_csv('/kaggle/working/Proyecto_Final_PINNs/rightWall_temporal_puntos.csv',index = False)

datos_lowerWall = pd.read_csv('/kaggle/working/Proyecto_Final_PINNs/lowerWall_temporal_puntos.csv')

datos_lowerWall['x'] = datos_lowerWall['x'] / L_ref
datos_lowerWall['y'] = datos_lowerWall['y'] / L_ref
datos_lowerWall['t'] = datos_lowerWall['t'] / T_ref

datos_lowerWall.to_csv('/kaggle/working/Proyecto_Final_PINNs/lowerWall_temporal_puntos.csv',index = False)

datos_atmosphere = pd.read_csv('/kaggle/working/Proyecto_Final_PINNs/atmosphere_temporal_puntos.csv')

datos_atmosphere['x'] = datos_atmosphere['x'] / L_ref
datos_atmosphere['y'] = datos_atmosphere['y'] / L_ref
datos_atmosphere['t'] = datos_atmosphere['t'] / T_ref

datos_atmosphere.to_csv('/kaggle/working/Proyecto_Final_PINNs/atmosphere_temporal_puntos.csv',index = False)

In [9]:
datos1 = pd.read_csv('/kaggle/working/Proyecto_Final_PINNs/datos_validation_primero.csv')
datos2 = pd.read_csv('/kaggle/working/Proyecto_Final_PINNs/datos_validation_segundo.csv')

datos = pd.concat([datos1,datos2])

datos.to_csv('/kaggle/working/Proyecto_Final_PINNs/datos_validation.csv',index = False)

In [11]:
def downsample_data_lhs(df, n_samples, columns=['Time', 'x', 'y']):
    """
    Submuestrea un DataFrame seleccionando puntos del set original que son
    los vecinos más cercanos a una grilla generada por Latin Hypercube.
    """
    print(f"Datos originales: {len(df)} puntos. Submuestreando a ~{n_samples}...")
    
    # 1. Extraer coordenadas de los datos reales
    # Asegúrate de que el orden sea consistente (ej: t, x, y)
    data_coords = df[columns].values
    
    # 2. Definir los límites del dominio basados en los datos
    # (t_min, x_min, y_min) y (t_max, x_max, y_max)
    min_bounds = data_coords.min(axis=0)
    max_bounds = data_coords.max(axis=0)
    
    # 3. Generar puntos objetivo con LHS
    # Generamos un poco más (e.g. 5-10%) porque algunos vecinos podrían repetirse
    n_lhs = int(n_samples * 1.1) 
    sampler = LatinHypercube(d=len(columns), seed=42)
    sample = sampler.random(n=n_lhs)
    
    # Escalar los puntos LHS al dominio físico de tus datos
    lhs_points = scale(sample, min_bounds, max_bounds)
    
    # 4. Buscar los vecinos más cercanos en los datos reales
    # Construir un KDTree con los datos reales para búsqueda rápida
    tree = cKDTree(data_coords)
    
    # Encontrar el índice del punto real más cercano a cada punto LHS
    # k=1 devuelve la distancia y el índice
    _, indices = tree.query(lhs_points, k=1)
    
    # 5. Seleccionar índices únicos y recortar al número deseado
    unique_indices = np.unique(indices)
    
    # Si obtuvimos más de los necesarios, recortamos aleatoriamente o los primeros
    if len(unique_indices) > n_samples:
        # Para mantener la distribución, un shuffle simple está bien
        np.random.shuffle(unique_indices)
        unique_indices = unique_indices[:n_samples]
        
    print(f"Puntos finales seleccionados: {len(unique_indices)}")
    
    # Retornar el DataFrame filtrado
    return df.iloc[unique_indices].reset_index(drop=True)

In [12]:
#DOWNSAMPLEAR

n_total_samples = 15000

df = pd.read_csv('/kaggle/working/Proyecto_Final_PINNs/datos_validation.csv')

mask_interface = (df['alpha.water'] > 0.30) & (df['alpha.water'] < 0.70)
df_interface = df[mask_interface]
df_bulk = df[~mask_interface]
    
n_interface = len(df_interface)
print(f"Puntos en interfase (se mantienen): {n_interface}")
    
# 2. Calcular cuántos puntos quedan para el resto (bulk)
n_remaining = n_total_samples - n_interface
    
if n_remaining > 0 and len(df_bulk) > 0:
    # 3. Aplicar LHS solo al "bulk" (aire/agua pura)
    df_bulk_sampled = downsample_data_lhs(df_bulk, n_remaining, columns=['Time', 'x', 'y'])
        
        # 4. Combinar
    df_final = pd.concat([df_interface, df_bulk_sampled])
else:
    # Si la interfase ya es más grande que n_total, priorizamos interfase y recortamos el resto
    print("Advertencia: La interfase ocupa casi todo el presupuesto de puntos.")
    if n_remaining < 0:
            # Opcional: Submuestrear también la interfase si es inmensa
        df_final = downsample_data_lhs(df_interface, n_total_samples) 
    else:
        df_final = df_interface

    # Mezclar aleatoriamente para que no queden ordenados
datos = df_final.sample(frac=1).reset_index(drop=True)
datos.to_csv('/kaggle/working/Proyecto_Final_PINNs/datos_validation_downsampled.csv')

Puntos en interfase (se mantienen): 7362
Datos originales: 455310 puntos. Submuestreando a ~7638...
Puntos finales seleccionados: 7638


In [13]:
def get_bump_profile(x, y, x_c, y_c, W_x, W_y, H_b, smoothness=0.01):
    """
    Crea un perfil de obstáculo "cuasi-cuadrado" suavizado usando tanh.

    Parámetros:
    x, y: Tensores de coordenadas
    x_c, y_c: Centro del obstáculo
    W_x, W_y: Ancho del obstáculo
    H_b: Altura máxima del obstáculo
    smoothness: Controla la agudeza de los bordes (un valor más pequeño es más "cuadrado")
    """

    # Perfil en X: (tanh(borde_izquierdo) - tanh(borde_derecho)) / 2
    # Esto crea un pulso suave que va de 0 a 1 y vuelve a 0.
    edge_x1 = torch.tanh((x - (x_c - W_x / 2)) / smoothness)
    edge_x2 = torch.tanh((x - (x_c + W_x / 2)) / smoothness)
    profile_x = (edge_x1 - edge_x2) / 2.0

    # Perfil en Y:
    edge_y1 = torch.tanh((y - (y_c - W_y / 2)) / smoothness)
    edge_y2 = torch.tanh((y - (y_c + W_y / 2)) / smoothness)
    profile_y = (edge_y1 - edge_y2) / 2.0

    # El perfil 2D es el producto de los perfiles 1D
    h0 = H_b * profile_x * profile_y
    return h0

In [14]:
##ARQUITECTURA DE PINNS

class PINN_Net(nn.Module):
    def __init__(self, layer_mat, device, act):
        super(PINN_Net, self).__init__()

        self.device = device
        self.layer_num = len(layer_mat) - 1
        self.act = act

        self.Re = Re_ref
        self.rho_ratio_aire = 1.2 / 998.0
        self.mu_ratio_aire = 1.81e-5 / 1.002e-3

        # Define separate networks for u, v, and p
        self.u_net = self._build_network(layer_mat[:-1] + [1]) # Output dimension 1 for u
        self.v_net = self._build_network(layer_mat[:-1] + [1]) # Output dimension 1 for v
        self.p_net = self._build_network(layer_mat[:-1] + [1]) # Output dimension 1 for p
        self.sig_xx_net = self._build_network(layer_mat[:-1] + [1]) # Output dimension 1 for sig_xx
        self.sig_yy_net = self._build_network(layer_mat[:-1] + [1]) # Output dimension 1 for sig_yy
        self.sig_xy_net = self._build_network(layer_mat[:-1] + [1]) # Output dimension 1 for sig_xy
        self.water_net = self._build_network(layer_mat[:-1] + [1]) # Output dimension 1 for alpha.water

        self.log_lambda_data = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))
        self.log_lambda_pde = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))
        self.log_lambda_bc = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))
        self.log_lambda_ic = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))

    def _build_network(self, layer_mat):
        """Helper function to build a sequential network."""
        network = nn.Sequential()
        for i in range(0, len(layer_mat) - 2):
            network.add_module(str(i) + "linear", nn.Linear(layer_mat[i], layer_mat[i + 1]))
            if self.act == 'tanh':
                network.add_module(str(i) + "Act", nn.Tanh())
            elif self.act == 'relu':
                network.add_module(str(i) + "Act", nn.ReLU())
            elif self.act == 'sigmoid':
                network.add_module(str(i) + "Act", nn.Sigmoid())

        network.add_module(str(len(layer_mat) - 2) + "linear",
                           nn.Linear(layer_mat[len(layer_mat) - 2], layer_mat[len(layer_mat) - 1]))
        self._initial_param(network) # Initialize parameters for this sub-network
        return network

    def forward(self, x, y, t):
        X = torch.cat([x, y, t], 1).requires_grad_(True).to(self.device) # Move input to device

        u_predict = self.u_net(X)
        v_predict = self.v_net(X)
        p_predict = self.p_net(X)
        sig_xx_predict = self.sig_xx_net(X)
        sig_yy_predict = self.sig_yy_net(X)
        sig_xy_predict = self.sig_xy_net(X)
        water_predict = torch.sigmoid(self.water_net(X))

        # Combine predictions if needed for some losses, but return separate for others
        return u_predict, v_predict, p_predict, sig_xx_predict, sig_yy_predict, sig_xy_predict, water_predict

    # initialize
    def _initial_param(self, network):
        for name, param in network.named_parameters():
            if name.endswith('linear.weight'):
                nn.init.xavier_normal_(param)
            elif name.endswith('linear.bias'):
                nn.init.zeros_(param)

    # derive loss for data
    def data_error(self, x, y, t, u, v, p,water):
        u_predict, v_predict, p_predict,_,_,_,water_predict = self.forward(x, y, t)
        mse = torch.nn.MSELoss()
        mse_predict = mse(u_predict, u) + mse(v_predict, v) + mse(p_predict, p) + mse(water_predict, water)
        mae = torch.nn.L1Loss()
        mae_predict = mae(u_predict, u) + mae(v_predict, v) + mae(p_predict, p) + mae(water_predict, water)
        return mse_predict, mae_predict



    def equation_error_espigon(self, x, y, t):
        # predict u, v, p at the obstacle points
        u, v, p, _,_,_, water = self.forward(x, y, t)

        mse = torch.nn.MSELoss()
        mae = torch.nn.L1Loss()

        # Calculate the MSE between predicted values and zero
        mse_u_espigon = mse(u, torch.zeros_like(u))
        mse_v_espigon = mse(v, torch.zeros_like(v))
        mse_p_espigon = mse(p, torch.zeros_like(p))
        mse_water_espigon = mse(water, torch.zeros_like(water))

        mae_u_espigon = mae(u, torch.zeros_like(u))
        mae_v_espigon = mae(v, torch.zeros_like(v))
        mae_p_espigon = mae(p, torch.zeros_like(p))
        mae_water_espigon = mae(water, torch.zeros_like(water))

        # Sum the losses
        mse_espigon = mse_u_espigon + mse_v_espigon + mse_p_espigon + mse_water_espigon
        mae_espigon = mae_u_espigon + mae_v_espigon + mae_p_espigon + mae_water_espigon
        return mse_espigon, mae_espigon


    # derive loss for equation
    def equation_error_dimensionless(self, x, y, t):
        x.requires_grad_(True)
        y.requires_grad_(True)
        t.requires_grad_(True)

        u, v, p, sig_xx, sig_yy, sig_xy, water = self.forward(x, y, t)


        rho_tilde = water * 1.0 + (1.0 - water) * self.rho_ratio_aire
        mu_tilde = water * 1.0 + (1.0 - water) * self.mu_ratio_aire

        # Derivadas (sin cambios)
        u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
        u_y = torch.autograd.grad(u.sum(), y, create_graph=True)[0]
        u_t = torch.autograd.grad(u.sum(), t, create_graph=True)[0]
        v_x = torch.autograd.grad(v.sum(), x, create_graph=True)[0]
        v_y = torch.autograd.grad(v.sum(), y, create_graph=True)[0]
        v_t = torch.autograd.grad(v.sum(), t, create_graph=True)[0]
        p_x = torch.autograd.grad(p.sum(), x, create_graph=True)[0]
        p_y = torch.autograd.grad(p.sum(), y, create_graph=True)[0]

        water_x = torch.autograd.grad(water.sum(), x, create_graph=True)[0]
        water_y = torch.autograd.grad(water.sum(), y, create_graph=True)[0]
        water_t = torch.autograd.grad(water.sum(), t, create_graph=True)[0]

        sig_xx_x = torch.autograd.grad(sig_xx.sum(), x, create_graph=True)[0]
        sig_yy_y = torch.autograd.grad(sig_yy.sum(), y, create_graph=True)[0]
        sig_xy_x = torch.autograd.grad(sig_xy.sum(), x, create_graph=True)[0]
        sig_xy_y = torch.autograd.grad(sig_xy.sum(), y, create_graph=True)[0]

        f_adveccion = water_t + u * water_x + v * water_y

        # Ecuaciones de Momento (usando rho_tilde)
        f_equation_x = rho_tilde * (u_t + u * u_x + v * u_y) + p_x - sig_xx_x - sig_xy_y
        f_equation_y = rho_tilde * (v_t + u * v_x + v * v_y) + p_y - sig_xy_x - sig_yy_y

        # Ecuación de Continuidad
        f_equation_mass = u_x + v_y

        # Ecuaciones Constitutivas (usando mu_tilde y el Re_ref CONSTANTE)
        f_const_xx = (2.0 / self.Re) * mu_tilde * u_x - sig_xx
        f_const_yy = (2.0 / self.Re) * mu_tilde * v_y - sig_yy
        f_const_xy = (1.0 / self.Re) * mu_tilde * (u_y + v_x) - sig_xy

        # --- Cálculo de Pérdidas (MSE) ---
        mse = torch.nn.MSELoss()
        mae = torch.nn.L1Loss()
        batch_t_zeros = torch.zeros_like(x, dtype=torch.float32, device=self.device)

        mse_adv = mse(f_adveccion, batch_t_zeros)
        mse_mom_x = mse(f_equation_x, batch_t_zeros)
        mse_mom_y = mse(f_equation_y, batch_t_zeros)
        mse_mass = mse(f_equation_mass, batch_t_zeros)
        mse_const_xx = mse(f_const_xx, batch_t_zeros)
        mse_const_yy = mse(f_const_yy, batch_t_zeros)
        mse_const_xy = mse(f_const_xy, batch_t_zeros)

        mae_adv = mae(f_adveccion, batch_t_zeros)
        mae_mom_x = mae(f_equation_x, batch_t_zeros)
        mae_mom_y = mae(f_equation_y, batch_t_zeros)
        mae_mass = mae(f_equation_mass, batch_t_zeros)
        mae_const_xx = mae(f_const_xx, batch_t_zeros)
        mae_const_yy = mae(f_const_yy, batch_t_zeros)
        mae_const_xy = mae(f_const_xy, batch_t_zeros)

        # Suma de todas las pérdidas
        mse_equation = mse_adv + mse_mom_x + mse_mom_y + mse_mass + \
                       mse_const_xx + mse_const_yy + mse_const_xy
        mae_equation = mae_adv + mae_mom_x + mae_mom_y + mae_mass + \
                       mae_const_xx + mae_const_yy + mae_const_xy

        return mse_equation, mae_equation

    def lowerWall_error(self, x, y, t):
        x.requires_grad_(True)
        y.requires_grad_(True)
        t.requires_grad_(True)

        u, v, p, sig_xx, sig_yy, sig_xy, water = self.forward(x, y, t)

        mse = torch.nn.MSELoss()
        mae = torch.nn.L1Loss()
        batch_t_zeros = torch.zeros_like(x, dtype=torch.float32, device=self.device)

        # ------------------ U - NoSlip (U=0) La velocidad del fluido en la pared es cero. Esta es una condición de Dirichlet.
        #La pérdida es el error cuadrático medio entre la velocidad predicha y cero.
        mse_u = mse(u, torch.zeros_like(u))
        mse_v = mse(v, torch.zeros_like(v))

        mae_u = mae(u, torch.zeros_like(u))
        mae_v = mae(v, torch.zeros_like(v))

        # ------------------ p - fixedFluxPressure (value  uniform 0)
        # Para una pared horizontal, la normal es la dirección 'y'.
        # Calculamos la derivada de 'p' con respecto a 'y' usando diferenciación automática.
        # El objetivo es que esta derivada sea cero.
        p_y = torch.autograd.grad(p.sum(), y, create_graph=True)[0]
        mse_p = mse(p_y, torch.zeros_like(p_y))
        mae_p = mae(p_y, torch.zeros_like(p_y))

        # # ------------------ alpha.water - zeroGradient
        # # Similar a la presión, el gradiente normal de alpha.water debe ser cero.
        # # Calculamos la derivada de 'alpha' con respecto a 'y'.
        water_y = torch.autograd.grad(water.sum(), y, create_graph=True)[0]
        mse_alpha = mse(water_y, torch.zeros_like(water_y))
        mae_alpha = mae(water_y, torch.zeros_like(water_y))

        # Sumar todas las pérdidas para obtener la pérdida total de esta frontera
        mse_lowerWall = mse_u + mse_v + mse_p + mse_alpha
        mae_lowerWall = mae_u + mae_v + mae_p + mae_alpha

        return mse_lowerWall, mae_lowerWall

    def leftWall_error(self, x, y, t):
        x.requires_grad_(True)
        y.requires_grad_(True)
        t.requires_grad_(True)

        u, v, p, sig_xx, sig_yy, sig_xy, water = self.forward(x, y, t)

        mse = torch.nn.MSELoss()
        mae = torch.nn.L1Loss()
        batch_t_zeros = torch.zeros_like(x, dtype=torch.float32, device=self.device)

        # ------------------ U - NoSlip (U=0) La velocidad del fluido en la pared es cero. Esta es una condición de Dirichlet.
        #La pérdida es el error cuadrático medio entre la velocidad predicha y cero.
        mse_u = mse(u, torch.zeros_like(u))
        mse_v = mse(v, torch.zeros_like(v))
        mae_u = mae(u, torch.zeros_like(u))
        mae_v = mae(v, torch.zeros_like(v))

        # ------------------ p - fixedFluxPressure (value  uniform 0)
        # Para una pared vertical, la normal es la dirección 'x'.
        # Calculamos la derivada de 'p' con respecto a 'x' usando diferenciación automática.
        # El objetivo es que esta derivada sea cero.
        p_x = torch.autograd.grad(p.sum(), x, create_graph=True)[0]
        mse_p = mse(p_x, torch.zeros_like(p_x))
        mae_p = mae(p_x, torch.zeros_like(p_x))

        # # ------------------ alpha.water - zeroGradient
        # # Similar a la presión, el gradiente normal de alpha.water debe ser cero.
        # # Calculamos la derivada de 'alpha' con respecto a 'y'.
        water_x = torch.autograd.grad(water.sum(), x, create_graph=True)[0]
        mse_alpha = mse(water_x, torch.zeros_like(water_x))
        mae_alpha = mae(water_x, torch.zeros_like(water_x))

        # Sumar todas las pérdidas para obtener la pérdida total de esta frontera
        mse_leftWall = mse_u + mse_v + mse_p + mse_alpha
        mae_leftWall = mae_u + mae_v + mae_p + mae_alpha

        return mse_leftWall, mae_leftWall

    def rightWall_error(self, x, y, t):
        x.requires_grad_(True)
        y.requires_grad_(True)
        t.requires_grad_(True)

        u, v, p, sig_xx, sig_yy, sig_xy, water = self.forward(x, y, t)

        mse = torch.nn.MSELoss()
        mae = torch.nn.L1Loss()
        batch_t_zeros = torch.zeros_like(x, dtype=torch.float32, device=self.device)

        # ------------------ U - NoSlip (U=0) La velocidad del fluido en la pared es cero. Esta es una condición de Dirichlet.
        #La pérdida es el error cuadrático medio entre la velocidad predicha y cero.
        mse_u = mse(u, torch.zeros_like(u))
        mse_v = mse(v, torch.zeros_like(v))
        mae_u = mae(u, torch.zeros_like(u))
        mae_v = mae(v, torch.zeros_like(v))

        # ------------------ p - fixedFluxPressure (value  uniform 0)
        # Para una pared vertical, la normal es la dirección 'x'.
        # Calculamos la derivada de 'p' con respecto a 'x' usando diferenciación automática.
        # El objetivo es que esta derivada sea cero.
        p_x = torch.autograd.grad(p.sum(), x, create_graph=True)[0]
        mse_p = mse(p_x, torch.zeros_like(p_x))
        mae_p = mae(p_x, torch.zeros_like(p_x))

        # # ------------------ alpha.water - zeroGradient
        # # Similar a la presión, el gradiente normal de alpha.water debe ser cero.
        # # Calculamos la derivada de 'alpha' con respecto a 'y'.
        water_x = torch.autograd.grad(water.sum(), x, create_graph=True)[0]
        mse_alpha = mse(water_x, torch.zeros_like(water_x))
        mae_alpha = mae(water_x, torch.zeros_like(water_x))
        # Sumar todas las pérdidas para obtener la pérdida total de esta frontera
        mse_rightWall = mse_u + mse_v + mse_p + mse_alpha
        mae_rightWall = mae_u + mae_v + mae_p + mae_alpha

        return mse_rightWall, mae_rightWall

    def atmosphere_error(self, x, y, t):
        x.requires_grad_(True)
        y.requires_grad_(True)
        t.requires_grad_(True)

        u, v, p, sig_xx, sig_yy, sig_xy, water = self.forward(x, y, t)

        mse = torch.nn.MSELoss()
        mae = torch.nn.L1Loss()
        batch_t_zeros = torch.zeros_like(x, dtype=torch.float32, device=self.device)

        # ------------------ U
        # Para una frontera horizontal, la normal es la dirección 'y'.
        # Calculamos las derivadas de 'u' y 'v' con respecto a 'y'.
        # El objetivo es que estas derivadas sean cero.
        u_y = torch.autograd.grad(u.sum(), y, create_graph=True)[0]
        mse_u = mse(u_y, torch.zeros_like(u_y))
        mae_u = mae(u_y, torch.zeros_like(u_y))

        v_y = torch.autograd.grad(v.sum(), y,create_graph=True)[0]
        mse_v = mse(v_y, torch.zeros_like(v_y))
        mae_v = mae(v_y, torch.zeros_like(v_y))

        # ------------------ p
        # Se fuerza a que la presión predicha sea cero.
        mse_p = mse(p, torch.zeros_like(p))
        mae_p = mae(p, torch.zeros_like(p))

        # # ------------------ alpha.water
        # # Se fuerza a que la fracción de fase sea cero
        # mse_alpha = mse(alpha, torch.zeros_like(alpha))

        # Sumar todas las pérdidas para obtener la pérdida total de esta frontera
        mse_atmosphere = mse_u + mse_v + mse_p #+ mse_alpha
        mae_atmosphere = mae_u + mae_v + mae_p #+ mae_alpha
        return mse_atmosphere, mae_atmosphere

    def initial_error(self, x, y, t):
        x.requires_grad_(True)
        y.requires_grad_(True)
        t.requires_grad_(True)

        u, v, p, sig_xx, sig_yy, sig_xy, water = self.forward(x, y, t)
        mse = torch.nn.MSELoss()
        mae = torch.nn.L1Loss()
        batch_t_zeros = torch.zeros_like(x, dtype=torch.float32, device=self.device)

        mse_u = mse(u, batch_t_zeros)
        mae_u = mae(u, batch_t_zeros)

        mse_v = mse(v, batch_t_zeros)
        mae_v = mae(v, batch_t_zeros)

        target_alpha = self.get_bump_profile(
            x, y, 
            x_c=0.073,   # Centro X
            y_c=0.146,    # Centro Y
            W_x=0.146,    # Ancho X
            W_y=0.292,    # Ancho Y
            H_b=1.0,     # Altura (Alpha = 1 para agua)
            smoothness=0.01 # Ajustar si es muy difuso o muy abrupto
        )

        mse_water = mse(water,target_alpha)
        mae_water = mae(water,target_alpha)

        mse_ic = mse_u + mse_v + mse_water
        mae_ic = mae_u + mae_v + mae_water
        return mse_ic, mae_ic
        

In [28]:
##FUNCIONES PARA CARGAR DATOS

# load data points ---> Puntos obtenidos de la simulación!!!!!!!!!!!!
def load_data_points(filename_data,boundary_dataframe):
    # Load data points
    data_df = pd.read_csv(filename_data)
    # Assuming the CSV has columns 'x', 'y', 't', 'Ux', 'Uy', 'p'
    x = data_df['x'].values.reshape(-1, 1)
    y = data_df['y'].values.reshape(-1, 1)
    t = data_df['Time'].values.reshape(-1, 1) # Assuming 'Time' column in CSV corresponds to 't'
    u = data_df['Ux'].values.reshape(-1, 1)
    v = data_df['Uy'].values.reshape(-1, 1)
    p = data_df['p'].values.reshape(-1, 1)
    water = data_df['alpha.water'].values.reshape(-1, 1)


    # Boundary conditions
    datos_atmosphere =  pd.read_csv(boundary_dataframe['atmosphere'])
    x_atmosphere = datos_atmosphere.loc[:, 'x'].values.reshape(-1, 1)
    y_atmosphere = datos_atmosphere.loc[:, 'y'].values.reshape(-1, 1)
    t_atmosphere = datos_atmosphere.loc[:, 't'].values.reshape(-1, 1)

    datos_leftWall = pd.read_csv(boundary_dataframe['leftWall'])
    x_leftWall = datos_leftWall.loc[:, 'x'].values.reshape(-1, 1)
    y_leftWall = datos_leftWall.loc[:, 'y'].values.reshape(-1, 1)
    t_leftWall = datos_leftWall.loc[:, 't'].values.reshape(-1, 1)

    datos_rightWall = pd.read_csv(boundary_dataframe['rightWall'])
    x_rightWall = datos_rightWall.loc[:, 'x'].values.reshape(-1, 1)
    y_rightWall = datos_rightWall.loc[:, 'y'].values.reshape(-1, 1)
    t_rightWall = datos_rightWall.loc[:, 't'].values.reshape(-1, 1)

    datos_lowerWall = pd.read_csv(boundary_dataframe['lowerWall'])
    x_lowerWall = datos_lowerWall.loc[:, 'x'].values.reshape(-1, 1)
    y_lowerWall = datos_lowerWall.loc[:, 'y'].values.reshape(-1, 1)
    t_lowerWall = datos_lowerWall.loc[:, 't'].values.reshape(-1, 1)

    x_ts = torch.tensor(x, dtype=torch.float32)
    y_ts = torch.tensor(y, dtype=torch.float32)
    t_ts = torch.tensor(t, dtype=torch.float32)
    u_ts = torch.tensor(u, dtype=torch.float32)
    v_ts = torch.tensor(v, dtype=torch.float32)
    p_ts = torch.tensor(p, dtype=torch.float32)
    water_ts = torch.tensor(water, dtype=torch.float32)


    x_atmosphere_ts = torch.tensor(x_atmosphere, dtype=torch.float32)
    y_atmosphere_ts = torch.tensor(y_atmosphere, dtype=torch.float32)
    t_atmosphere_ts = torch.tensor(t_atmosphere, dtype=torch.float32)

    x_leftWall_ts = torch.tensor(x_leftWall, dtype=torch.float32)
    y_leftWall_ts = torch.tensor(y_leftWall, dtype=torch.float32)
    t_leftWall_ts = torch.tensor(t_leftWall, dtype=torch.float32)

    x_rightWall_ts = torch.tensor(x_rightWall, dtype=torch.float32)
    y_rightWall_ts = torch.tensor(y_rightWall, dtype=torch.float32)
    t_rightWall_ts = torch.tensor(t_rightWall, dtype=torch.float32)

    x_lowerWall_ts = torch.tensor(x_lowerWall, dtype=torch.float32)
    y_lowerWall_ts = torch.tensor(y_lowerWall, dtype=torch.float32)
    t_lowerWall_ts = torch.tensor(t_lowerWall, dtype=torch.float32)


    return x_ts, y_ts, t_ts, u_ts, v_ts, p_ts,water_ts, x_atmosphere_ts, y_atmosphere_ts, t_atmosphere_ts, x_leftWall_ts, y_leftWall_ts, t_leftWall_ts, x_rightWall_ts, y_rightWall_ts, t_rightWall_ts, x_lowerWall_ts, y_lowerWall_ts, t_lowerWall_ts

def load_equation_points_lhs(dimension, points):
  """
    Genera puntos de colocación usando LHS, excluyendo una
    región de obstáculo (el espigón).

    Args:
        dimension (int): Número de dimensiones (debe ser 3 para t, x, y).
        points (int): Número final de puntos válidos deseados.
    """

    # --- 1. Definir Límites ---
  t_min, t_max = 0.0/ T_ref, 4.9997 / T_ref

  # Límites del Bounding Box (el cuadrado grande)
  low_bounds = [t_min, 0.001 / L_ref, 0.001 / L_ref]     # [t_min, x_min, y_min]
  up_bounds = [t_max, 0.583 / L_ref , 0.583 / L_ref]      # [t_max, x_max, y_max]

  # Límites del Obstáculo (espigón)
  obs_x_min, obs_x_max = 0.291 / L_ref, 0.315 / L_ref
  obs_y_min, obs_y_max = 0.001/ L_ref, 0.047/ L_ref

  # --- 2. Oversampling ---
  # El área del obstáculo es ~0.3% del área total.
  # Generar un 5% extra de puntos (1.05) es más que suficiente
  # para garantizar que tendremos 'points' puntos válidos.
  N_to_generate = int(points * 1.05)

  # Asegurarnos de generar al menos algunos extra
  if N_to_generate <= points:
      N_to_generate = points + 50

  # --- 3. Generación LHS (en Numpy) ---
  if dimension != 3:
      print(f"Advertencia: La dimensión es {dimension}, pero se esperan 3 (t, x, y)")

  sampler = LatinHypercube(d=dimension, seed=42) # seed para reproducibilidad
  sample_normalized = sampler.random(n=N_to_generate)

  # Escalar puntos al bounding box
  eqa_xyzt_all = scale(sample_normalized, low_bounds, up_bounds)

  # --- 4. Convertir a Torch ---
  # Convertimos a Torch AHORA para hacer el filtrado con tensores.
  Eqa_points_all = torch.from_numpy(eqa_xyzt_all).float()

  # --- 5. Filtrar el Obstáculo (¡Este es el paso clave!) ---
  # Extraemos las coordenadas X e Y de todos los puntos generados
  x_all = Eqa_points_all[:, 1]
  y_all = Eqa_points_all[:, 2]

  # Crear una máscara booleana para los puntos DENTRO del obstáculo
  is_inside_obstacle = (
      (x_all >= obs_x_min) & (x_all <= obs_x_max) &
      (y_all >= obs_y_min) & (y_all <= obs_y_max)
  )

  # Invertir la máscara para quedarnos solo con los puntos VÁLIDOS (fuera)
  is_valid_point = ~is_inside_obstacle

  # Aplicar el filtro
  Eqa_points_valid = Eqa_points_all[is_valid_point]
  Eqa_points_espigon = Eqa_points_all[is_inside_obstacle] # Keep points inside obstacle for equation loss calculation

  # --- 6. Recortar al número deseado y Mezclar --

  # Ahora aplicamos el shuffle (mezcla) que tenías en tu línea original
  Eqa_points_valid = Eqa_points_valid[torch.randperm(Eqa_points_valid.size(0))]
  Eqa_points_espigon = Eqa_points_espigon[torch.randperm(Eqa_points_espigon.size(0))]

  return Eqa_points_valid, Eqa_points_espigon


# split batch and automatically fill batch size
# batch 划分与自动填充
def batch_split(Set, iter_num, dim=0):
    batches = torch.chunk(Set, iter_num, dim=dim)
    # 自动填充
    num_of_batches = len(batches)
    if num_of_batches == 1:
        batches = batches * iter_num
        return batches
    if num_of_batches < iter_num:
        for i in range(iter_num - num_of_batches):
            index = i % num_of_batches
            add_tuple = batches[-(index + 2):-(index + 1)]
            batches = batches + add_tuple
        return batches
    else:
        return batches


# preprocessing before training
def pre_train_loading(filename_data, boundary_dataframe, dimension, number_eqa, batch_size):
    # load data points(only once)
    x_ts, y_ts, t_ts, u_ts, v_ts, p_ts, water_ts, x_atmosphere_ts, y_atmosphere_ts, t_atmosphere_ts, x_leftWall_ts, y_leftWall_ts, t_leftWall_ts, x_rightWall_ts, y_rightWall_ts, t_rightWall_ts, x_lowerWall_ts, y_lowerWall_ts, t_lowerWall_ts = load_data_points(filename_data, boundary_dataframe)

    # si hay datos......
    if x_ts.shape[0] > 0:
        # Esto crea un único tensor donde cada fila representa un punto de datos
        # con sus coordenadas y valores correspondientes para (u, v, p)
        data_sub = torch.cat([x_ts, y_ts, t_ts, u_ts, v_ts, p_ts, water_ts], 1)
        # Mezcla aleatoriamente las filas del tensor data_sub
        true_dataset = data_sub[torch.randperm(data_sub.size(0))]

        # Ahora para las CB
        atmosphere_dataset = torch.cat([x_atmosphere_ts, y_atmosphere_ts, t_atmosphere_ts], 1)
        leftWall_dataset = torch.cat([x_leftWall_ts, y_leftWall_ts, t_leftWall_ts] ,1)
        rightWall_dataset = torch.cat([x_rightWall_ts, y_rightWall_ts, t_rightWall_ts], 1)
        lowerWall_dataset = torch.cat([x_lowerWall_ts, y_lowerWall_ts, t_lowerWall_ts], 1)

        # Mezcla aleatoriamente las filas de los tensor
        atmosphere_dataset = atmosphere_dataset[torch.randperm(atmosphere_dataset.size(0))]
        leftWall_dataset = leftWall_dataset[torch.randperm(leftWall_dataset.size(0))]
        rightWall_dataset = rightWall_dataset[torch.randperm(rightWall_dataset.size(0))]
        lowerWall_dataset = lowerWall_dataset[torch.randperm(lowerWall_dataset.size(0))]

    else:
        #si no, no usa data
        true_dataset = None

    # load collocation points(only once)
    eqa_points_valid, eqa_points_espigon = load_equation_points_lhs(dimension, number_eqa)
    eqa_points_batches = torch.split(eqa_points_valid, batch_size, dim=0)
    iter_num = len(eqa_points_batches)
    if true_dataset is not None:
        true_dataset_batches = batch_split(true_dataset, iter_num)
    else:
        true_dataset_batches = [None] * iter_num

    atmosphere_dataset_batches = batch_split(atmosphere_dataset, iter_num)
    leftWall_dataset_batches = batch_split(leftWall_dataset, iter_num)
    rightWall_dataset_batches = batch_split(rightWall_dataset, iter_num)
    lowerWall_dataset_batches = batch_split(lowerWall_dataset, iter_num)
                                                                                                    # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    return true_dataset_batches, eqa_points_batches, eqa_points_espigon, iter_num, atmosphere_dataset, leftWall_dataset, rightWall_dataset, lowerWall_dataset
    #acá devuelve true_dataset en lugar de los batches

# train data points, collocation points--with batch training
def train_data_whole(inner_epochs, pinn_example, optimizer_all, scheduler_all, iter_num, true_dataset_batches,
                     Eqa_points_batches, Eqa_points_espigon, atmosphere_dataset, leftWall_dataset, rightWall_dataset, lowerWall_dataset,
                    weight_data, weight_eqa, weight_atmosphere, weight_leftWall, weight_rightWall, weight_lowerWall, EPOCH, debug_key, device,dynamic_weights,
                    flag_data, flag_eqa,flag_bc,MSE):

    loss_tracker = {
        'total': 0.0, 'data': 0.0, 'eqa': 0.0, 
        'bc_atmosphere': 0.0, 'bc_left': 0.0, 'bc_right': 0.0, 'bc_lower': 0.0,
        'mae_data': 0.0, 'mae_eqa': 0.0
    }

    x_atmosphere_train = atmosphere_dataset[:, 0].reshape(-1, 1).requires_grad_(True).to(device)
    y_atmosphere_train = atmosphere_dataset[:, 1].reshape(-1, 1).requires_grad_(True).to(device)
    t_atmosphere_train = atmosphere_dataset[:, 2].reshape(-1, 1).requires_grad_(True).to(device)

    x_leftWall_train = leftWall_dataset[:, 0].reshape(-1, 1).requires_grad_(True).to(device)
    y_leftWall_train = leftWall_dataset[:, 1].reshape(-1, 1).requires_grad_(True).to(device)
    t_leftWall_train = leftWall_dataset[:, 2].reshape(-1, 1).requires_grad_(True).to(device)

    x_rightWall_train = rightWall_dataset[:, 0].reshape(-1, 1).requires_grad_(True).to(device)
    y_rightWall_train = rightWall_dataset[:, 1].reshape(-1, 1).requires_grad_(True).to(device)
    t_rightWall_train = rightWall_dataset[:, 2].reshape(-1, 1).requires_grad_(True).to(device)
    
    x_lowerWall_train = lowerWall_dataset[:, 0].reshape(-1, 1).requires_grad_(True).to(device)
    y_lowerWall_train = lowerWall_dataset[:, 1].reshape(-1,1).requires_grad_(True).to(device)
    t_lowerWall_train = lowerWall_dataset[:, 2].reshape(-1,1).requires_grad_(True).to(device)
    
    x_espigon = Eqa_points_espigon[:, 1].reshape(-1, 1).requires_grad_(True).to(device)
    y_espigon = Eqa_points_espigon[:, 2].reshape(-1, 1).requires_grad_(True).to(device)
    t_espigon = Eqa_points_espigon[:, 0].reshape(-1, 1).requires_grad_(True).to(device)
    

    for epoch in range(inner_epochs):
        for batch_iter in range(iter_num):
            true_dataset_batch = true_dataset_batches[batch_iter]
            x_train, y_train, t_train, u_train, v_train, p_train, water_train = [None]*7
            
            if true_dataset_batch is not None:
                x_train = true_dataset_batch[:, 0].reshape(-1, 1).requires_grad_(True).to(device)
                y_train = true_dataset_batch[:, 1].reshape(-1, 1).requires_grad_(True).to(device)
                t_train = true_dataset_batch[:, 2].reshape(-1, 1).requires_grad_(True).to(device)
                u_train = true_dataset_batch[:, 3].reshape(-1, 1).to(device)
                v_train = true_dataset_batch[:, 4].reshape(-1, 1).to(device)
                p_train = true_dataset_batch[:, 5].reshape(-1, 1).to(device)
                water_train = true_dataset_batch[:, 6].reshape(-1, 1).to(device)

            x_eqa = Eqa_points_batches[batch_iter][:, 1].reshape(-1, 1).requires_grad_(True).to(device)
            y_eqa = Eqa_points_batches[batch_iter][:, 2].reshape(-1, 1).requires_grad_(True).to(device)
            t_eqa = Eqa_points_batches[batch_iter][:, 0].reshape(-1, 1).requires_grad_(True).to(device)

            # --- Definición del Closure ---
            def closure():
                optimizer_all.zero_grad()
                
                # 1. Data Loss
                if true_dataset_batch is not None:
                    mse_data, mae_data = pinn_example.data_error(x_train, y_train, t_train, u_train, v_train, p_train, water_train)
                else:
                    mse_data = torch.tensor(0.0, device=device)
                    mae_data = torch.tensor(0.0, device=device)

                # 2. Equation Loss

                mse_equation, mae_equation = pinn_example.equation_error_dimensionless(x_eqa, y_eqa, t_eqa)
                mse_espigon, mae_espigon = pinn_example.equation_error_espigon(x_espigon, y_espigon, t_espigon)

                mse_equation = 0.95 * mse_equation + 0.05 * mse_espigon
                mae_equation = 0.95 * mae_equation + 0.05 * mae_espigon

                # 3. Boundary Condition Loss
                mse_atmosphere, mae_atmosphere = pinn_example.atmosphere_error(x_atmosphere_train, y_atmosphere_train, t_atmosphere_train)
                mse_leftWall, mae_leftWall = pinn_example.leftWall_error(x_leftWall_train, y_leftWall_train, t_leftWall_train)
                mse_rightWall, mae_rightWall = pinn_example.rightWall_error(x_rightWall_train, y_rightWall_train, t_rightWall_train)
                mse_lowerWall, mae_lowerWall = pinn_example.lowerWall_error(x_lowerWall_train, y_lowerWall_train, t_lowerWall_train)
                # mse_ic,mae_ic = pinn_example.initial_error()

                # 4. Total Loss Calculation
                if dynamic_weights:
                    mse_bc = (mse_atmosphere + mse_leftWall + mse_rightWall + mse_lowerWall) / 4
                    mae_bc = (mae_atmosphere + mae_leftWall + mae_rightWall + mae_lowerWall) / 4
                    loss_data = torch.exp(-pinn_example.log_lambda_data) * mse_data * MSE + torch.exp(-pinn_example.log_lambda_data) * mae_data * (not MSE) + pinn_example.log_lambda_data
                    loss_eqa = torch.exp(-pinn_example.log_lambda_pde) * mse_equation * MSE + torch.exp(-pinn_example.log_lambda_pde) * mae_equation * (not MSE) + pinn_example.log_lambda_pde
                    loss_bc = torch.exp(-pinn_example.log_lambda_bc) * mse_bc * MSE + torch.exp(-pinn_example.log_lambda_bc) * mae_bc * (not MSE) + pinn_example.log_lambda_bc
                    loss = loss_data * flag_data + loss_eqa * flag_eqa + loss_bc * flag_bc
                else:
                    #ESTO NO SE USA, PERO EN CASO DE QUERER USARLO HAY QUE MODIFICARLO
                    loss_data_term = weight_data * mse_data
                    loss_eqa_term = weight_eqa * mse_equation
                    loss_bc_term = weight_atmosphere * mse_atmosphere + weight_leftWall * mse_leftWall + weight_rightWall * mse_rightWall + weight_lowerWall * mse_lowerWall
                    loss = loss_data_term * flag_data + loss_eqa_term * flag_eqa + loss_bc_term * flag_bc

                loss.backward()
                
                # Guardar valores para logging (detach para no guardar grafo)
                loss_tracker['total'] = loss.item()
                loss_tracker['data'] = mse_data.item()
                loss_tracker['eqa'] = mse_equation.item()
                loss_tracker['bc_right'] = mse_rightWall.item()
                loss_tracker['bc_lower'] = mse_lowerWall.item()
                loss_tracker['bc_left'] = mse_leftWall.item()
                loss_tracker['bc_atmosphere'] = mse_atmosphere.item()
                loss_tracker['mae_data'] = mae_data.item()
                loss_tracker['mae_eqa'] = mae_equation.item()
                
                return loss
            # --- Fin del Closure ---

            # Paso de optimización
            optimizer_all.step(closure)

            # Logging
            if (batch_iter + 1) % iter_num == 0 and debug_key == 1:
                 print(f"EPOCH: {EPOCH + 1} inner_iter: {batch_iter + 1} Total: {loss_tracker['total']:.6f} "
                       f"Data: {loss_tracker['data']:.6f} PDE: {loss_tracker['eqa']:.6f}")

        # Paso del scheduler (fuera del bucle de batches si es por época)
        if not isinstance(optimizer_all, torch.optim.LBFGS): # LBFGS suele manejar su propio LR o no usar scheduler estándar
            scheduler_all.step()

    # Retornar los últimos valores registrados
    return (np.array([loss_tracker['total']]), np.array([loss_tracker['data']]), 
            np.array([loss_tracker['eqa']]), np.array([loss_tracker['bc_atmosphere']]),
            np.array([loss_tracker['bc_left']]), np.array([loss_tracker['bc_right']]), 
            np.array([loss_tracker['bc_lower']]),np.array([loss_tracker['mae_data']]), np.array([loss_tracker['mae_eqa']]))
# record loss
# 记录loss
def record_loss_local(loss_sum, loss_data, loss_eqa, filename_loss):
    loss_sum_value = loss_sum.reshape(1, 1)
    loss_data_value = loss_data.reshape(1, 1)
    loss_eqa_value = loss_eqa.reshape(1, 1)
    loss_set = np.concatenate((loss_sum_value, loss_data_value, loss_eqa_value), 1).reshape(1, -1)
    loss_save = pd.DataFrame(loss_set)
    loss_save.to_csv(filename_loss, index=False, header=False, mode='a')
    return loss_set

In [23]:
##FUNCIONES PARA PARAMETROS DE LA RED
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

train_config = {
    'write_path': '/kaggle/working/write',
    'loss_path': '/kaggle/working/write/loss.csv',
    'evaluate_path': '/kaggle/working/write/RL2.csv',
    'data_path': '/kaggle/working/Proyecto_Final_PINNs/datos_validation_downsampled.csv',
    'real_path': '/kaggle/working/Proyecto_Final_PINNs/datos_validation_downsampled.csv',
    'atmosphere_path': '/kaggle/working/Proyecto_Final_PINNs/atmosphere_temporal_puntos.csv',
    'leftWall_path': '/kaggle/working/Proyecto_Final_PINNs/leftWall_temporal_puntos.csv',
    'rightWall_path': '/kaggle/working/Proyecto_Final_PINNs/rightWall_temporal_puntos.csv',
    'lowerWall_path': '/kaggle/working/Proyecto_Final_PINNs/lowerWall_temporal_puntos.csv',
    'dimension': 3,  # Evaluado de 2 + 1
    'outer_epoch': 5000,
    'inner_epochs': 1,
    'save_interval': 100,
    'number_eqa': 100000, # Reduced number of equation points
    'optimizer': 'adam', # Changed optimizer to Adam
    'scheduler': 'exp',
    'act': 'tanh',
    'batch_size': 10000,
    'learning_rate': 1e-3,
    'dynamic_weights' : True,
    'NavierRe': True,
    'hidden_layers': 12,
    'layer_neurons': 96,
    'weight_of_data': 1,
    'weight_of_eqa': 0.1,
    'weight_atmosphere': 0, #---------> no contribuye
    'weight_rightWall': 0,
    'weight_letfWall': 0,
    'weight_lowerWall': 0,
    'debug_key': 1,
    'flag_data': True,
    'flag_eqa': True,
    'flag_bc': True,
    'MSE':True,
    'switch_to_lbfgs_epoch': 1000, # Época donde cambia a L-BFGS. Pon -1 para no cambiar nunca.
    'lbfgs_max_iter': 20,          # Iteraciones máximas por paso de L-BFGS
    'lbfgs_history_size': 50,
}

def build_optimizer(network, optimizer_name, scheduler_name, learning_rate):
    # default 默认优化器
    optimizer = torch.optim.Adam(network.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.999)
    # 超参数搜索优化器
    if optimizer_name == "sgd":
        optimizer = torch.optim.SGD(network.parameters(), lr=learning_rate, momentum=0.9)
    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(network.parameters(), lr=learning_rate)
    elif optimizer_name == "L-BFGS":
        optimizer = torch.optim.LBFGS(
            network.parameters(), 
            lr=1.0, # L-BFGS suele usar lr=1, pero puedes ajustarlo
            max_iter=train_config.get('lbfgs_max_iter', 20), 
            max_eval=train_config.get('lbfgs_max_iter', 20) * 1.25, 
            history_size=train_config.get('lbfgs_history_size', 50),
            tolerance_grad=1e-5, 
            tolerance_change=1.0 * np.finfo(float).eps,
            line_search_fn="strong_wolfe" # Importante para buena convergencia en PINNs
        )
    if scheduler_name == "exp":
        scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.999)
    elif scheduler_name == "fix":
        scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=1)
    return optimizer, scheduler


def train():

    # important parameters
    # 重要参数
    data_path = train_config['data_path']
    dimension = train_config['dimension']
    real_path = train_config['real_path']
    write_path = train_config['write_path']
    evaluate_path = train_config['evaluate_path']
    loss_path = train_config['loss_path']
    inner_epochs = train_config['inner_epochs']
    save_interval = train_config['save_interval']
    outer_epochs = train_config['outer_epoch']
    number_eqa = train_config['number_eqa']
    debug_key = train_config['debug_key']

    layer_mat = [dimension] + train_config['hidden_layers'] * [train_config['layer_neurons']] + [3]
    learning_rate = train_config['learning_rate']
    batch_size = train_config['batch_size']
    weight_of_data = train_config['weight_of_data']
    weight_of_eqa = train_config['weight_of_eqa']
    weight_atmosphere = train_config['weight_atmosphere']
    weight_rightWall = train_config['weight_rightWall']
    weight_leftWall = train_config['weight_letfWall']
    weight_lowerWall = train_config['weight_lowerWall']
    optimizer_name = train_config['optimizer']
    scheduler_name = train_config['scheduler']
    act = train_config['act']
    flag_data = train_config['flag_data']
    flag_eqa = train_config['flag_eqa']
    flag_bc = train_config['flag_bc']
    boundary_dataframes = {
        'atmosphere': train_config['atmosphere_path'],
        'leftWall': train_config['leftWall_path'],
        'rightWall': train_config['rightWall_path'],
        'lowerWall': train_config['lowerWall_path']
    }
    NavierRe = train_config['NavierRe']
    dynamic_weights = train_config['dynamic_weights']
    MSE = train_config['MSE']
    wandb.init(
        project="damBreak-experimentos",  # El nombre de tu proyecto
        config=train_config         # 3. Guardar toda la configuración
    )
    True_dataset_batches, Eqa_points_batches,Eqa_points_espigon, iter_num, atmosphere_dataset, leftWall_dataset, rightWall_dataset, lowerWall_dataset = pre_train_loading(
        data_path,
        boundary_dataframes,
        dimension,
        number_eqa,
        batch_size)

    pinn_net = PINN_Net(layer_mat, device, act=act)
    pinn_net = pinn_net.to(device)
    
    


    # 优化器和学习率衰减设置- optimizer and learning rate schedule
    current_optimizer_name = optimizer_name
    optimizer_all, scheduler_all = build_optimizer(pinn_net, current_optimizer_name, scheduler_name, learning_rate)
    
    switch_epoch = train_config.get('switch_to_lbfgs_epoch', -1)
    
    if not os.path.exists(write_path):
        # 创建文件夹 create file for recording
        os.mkdir(write_path)
    # 训练主循环 main loop
    for EPOCH in range(outer_epochs):
        if EPOCH == switch_epoch and current_optimizer_name != "L-BFGS":
            print(f"--- Switching optimizer from {current_optimizer_name} to L-BFGS at epoch {EPOCH} ---")
            current_optimizer_name = "L-BFGS"
            # Reconstruir optimizador (L-BFGS usualmente no usa el mismo scheduler que Adam)
            optimizer_all, scheduler_all = build_optimizer(pinn_net, "L-BFGS", "fix", learning_rate)
            
        loss_sum, loss_data, loss_eqa, loss_atmosphere, loss_leftWall, loss_rightWall, loss_lowerWall, mae_data,mae_eqa = train_data_whole(inner_epochs, pinn_net, optimizer_all, scheduler_all,
                                                             iter_num, True_dataset_batches, Eqa_points_batches, Eqa_points_espigon,
                                                             atmosphere_dataset, leftWall_dataset, rightWall_dataset, lowerWall_dataset,
                                                             weight_of_data, weight_of_eqa, weight_atmosphere, weight_leftWall,
                                                             weight_rightWall, weight_lowerWall, EPOCH, debug_key,
                                                             device,dynamic_weights,flag_data,flag_eqa,flag_bc,MSE)

        wandb.log({
            "epoch": EPOCH + 1,
            "total_loss": loss_sum,
            "data_loss_mse": loss_data,
            "eqa_loss_mse": loss_eqa,
            "atmosphere_loss_mse": loss_atmosphere,
            "leftWall_loss_mse": loss_leftWall,
            "rightWall_loss_mse": loss_rightWall,
            "lowerWall_loss_mse": loss_lowerWall,
            "data_loss_mae" : mae_data,
            "eqa_loss_mae": mae_eqa,
            "learning_rate": optimizer_all.param_groups[0]['lr'] # ¡Útil para rastrear el scheduler!
        })

        # 每隔固定Epoch保存模型 save model at every save_interval epoch
        # 每隔固定Epoch评估模型 evaluate model at every save_interval epoch
        if not os.path.exists(write_path):
            os.makedirs(write_path)
        if ((EPOCH + 1) % save_interval == 0) | (EPOCH == 0):
            dir_name = write_path + '/step' + str(EPOCH + 1)
            os.makedirs(dir_name, exist_ok=True)
            torch.save(pinn_net.state_dict(), dir_name + '/NS_model_train.pt')
            print(f'Model saved at step {EPOCH + 1}.')
            # valid_u, valid_v, valid_p = validation(pinn_net, real_path, inner_Norm, evaluate_path)
            print(f'Model evaluated at step {EPOCH + 1}.')

    wandb.finish()
    return

cuda:0


In [47]:
wandb.finish()

In [ ]:

if __name__ == '__main__':
    train()

atmosphere_loss_mse,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
data_loss_mae,█▄▃▂▂▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
data_loss_mse,█▅▂▂▂▁▃▃▃▃▃▃▃▃▃▃▃▃▃▃▂▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃▃
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
eqa_loss_mae,█▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eqa_loss_mse,█▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▁▃▃▁▁▂▂▂▂▂▂▂▂▂▂▃▂▂▂▂▂▂
learning_rate,████▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁
leftWall_loss_mse,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
lowerWall_loss_mse,▇▇▅▁▂▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁██▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
rightWall_loss_mse,█▆▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_loss,█▇▇▇▇▆▆▆▆▆▅▅▄▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁


EPOCH: 1 inner_iter: 11 Total: 0.388741 Data: 0.244538 PDE: 0.087301
Model saved at step 1.
Model evaluated at step 1.
EPOCH: 2 inner_iter: 11 Total: 0.246669 Data: 0.169899 PDE: 0.053173
EPOCH: 3 inner_iter: 11 Total: 0.101341 Data: 0.112981 PDE: 0.020876
EPOCH: 4 inner_iter: 11 Total: 0.041568 Data: 0.096858 PDE: 0.009943
EPOCH: 5 inner_iter: 11 Total: 0.000461 Data: 0.089567 PDE: 0.006742
EPOCH: 6 inner_iter: 11 Total: -0.035623 Data: 0.083298 PDE: 0.006631
EPOCH: 7 inner_iter: 11 Total: -0.061035 Data: 0.090642 PDE: 0.006927
EPOCH: 8 inner_iter: 11 Total: -0.073716 Data: 0.112135 PDE: 0.002552
EPOCH: 9 inner_iter: 11 Total: -0.113571 Data: 0.101340 PDE: 0.003931
EPOCH: 10 inner_iter: 11 Total: -0.150779 Data: 0.091967 PDE: 0.004257
EPOCH: 11 inner_iter: 11 Total: -0.177736 Data: 0.096727 PDE: 0.003298
EPOCH: 12 inner_iter: 11 Total: -0.213001 Data: 0.088215 PDE: 0.004266
EPOCH: 13 inner_iter: 11 Total: -0.246258 Data: 0.081861 PDE: 0.005419
EPOCH: 14 inner_iter: 11 Total: -0.280633

In [2]:
##CREAR MAPA DE UN SOLO TIEMPO

# Define the path to the trained model file
model_path = '/kaggle/working/write/step1000/NS_model_train.pt' # Assuming the last saved model is at step 5000
# Define the network architecture
layer_mat = [train_config['dimension']] + train_config['hidden_layers'] * [train_config['layer_neurons']] + [7]

# Load the real data used for normalization (for Time, Ux, Uy, p)
# Note: Although the function is called load_data_points, it also loads other dataframes.
# We only need the main data for normalization here.

boundary_dataframes = {
    'atmosphere': train_config['atmosphere_path'],
    'leftWall': train_config['leftWall_path'],
    'rightWall': train_config['rightWall_path'],
    'lowerWall': train_config['lowerWall_path']
}
x_ts, y_ts, t_ts, u_ts, v_ts, p_ts,water_ts, x_atmosphere_ts, y_atmosphere_ts, t_atmosphere_ts, x_leftWall_ts, y_leftWall_ts, t_leftWall_ts, x_rightWall_ts, y_rightWall_ts, t_rightWall_ts, x_lowerWall_ts, y_lowerWall_ts, t_lowerWall_ts  = load_data_points(train_config['data_path'], boundary_dataframes)

# Create dummy mean and standard deviation numpy arrays for PINN_Net initialization
# These are not used for normalization when inner_Norm is 'no_norm', but the PINN_Net class requires them.
dummy_mean = np.zeros((1, train_config['dimension']))
dummy_std = np.ones((1, train_config['dimension']))


# Instantiate the PINN_Net model
pinn_net = PINN_Net(layer_mat, device, act=train_config['act'])
pinn_net = pinn_net.to(device)
# Load the state dictionary from the trained model file
pinn_net.load_state_dict(torch.load(model_path, map_location=device))
# Set the model to evaluation mode
pinn_net.eval()

# Load the real data for plotting
real_data_df = pd.read_csv(train_config['real_path'])

# Find the unique time steps in the original data
unique_times = datos['Time'].unique()

# Find the time step closest to 1.0
closest_time = unique_times[np.abs(unique_times - 1).argmin()]

print(f"Closest time step to 1.0 in the data: {closest_time}")

# Filter the real data to get the points at the closest time step
real_data_closest_time = real_data_df[real_data_df['Time'] == closest_time].copy()

# Extract the required columns from the filtered data at the closest time step
x_closest_time = real_data_closest_time['x'].values
y_closest_time = real_data_closest_time['y'].values
t_closest_time = real_data_closest_time['Time'].values

# Convert to tensors and reshape
x_ts_closest_time = torch.tensor(x_closest_time, dtype=torch.float32).reshape(-1, 1).to(device)
y_ts_closest_time = torch.tensor(y_closest_time, dtype=torch.float32).reshape(-1, 1).to(device)
t_ts_closest_time = torch.tensor(t_closest_time, dtype=torch.float32).reshape(-1, 1).to(device)

with torch.no_grad():
    # The PINN_Net.predict method takes x, y, and t (normalized) as input.
    u_pred_closest_time, v_pred_closest_time, p_pred_closest_time, sig_xx_pred_norm_closest_time, sig_yy_pred_norm_closest_time, \
    sig_xy_pred_norm_closest_time, water_pred_closest_time = pinn_net.forward(x_ts_closest_time, y_ts_closest_time, t_ts_closest_time)

# Prepare the real data for comparison at the closest time step
u_real_closest_time = torch.tensor(real_data_closest_time['Ux'].values, dtype=torch.float32).reshape(-1, 1).to(device)
v_real_closest_time = torch.tensor(real_data_closest_time['Uy'].values, dtype=torch.float32).reshape(-1, 1).to(device)
p_real_closest_time = torch.tensor(real_data_closest_time['p'].values, dtype=torch.float32).reshape(-1, 1).to(device)
water_real_closest_time = torch.tensor(real_data_closest_time['alpha.water'].values, dtype=torch.float32).reshape(-1, 1).to(device)

# Display the shapes of the denormalized predicted tensors and real data tensors
print("Shape of u_pred_closest_time:", u_pred_closest_time.shape)
print("Shape of v_pred_closest_time:", v_pred_closest_time.shape)
print("Shape of p_pred_closest_time:", p_pred_closest_time.shape)
print("Shape of u_real_closest_time:", u_real_closest_time.shape)
print("Shape of v_real_closest_time:", v_real_closest_time.shape)
print("Shape of p_real_closest_time:", p_real_closest_time.shape)
print("Shape of water_real_closest_time:", water_real_closest_time.shape)

# Create a meshgrid for the x and y coordinates
x_unique = np.unique(real_data_closest_time['x'])
y_unique = np.unique(real_data_closest_time['y'])
X_grid, Y_grid = np.meshgrid(x_unique, y_unique)

# Prepare the interpolation points (x, y coordinates from real data) and values (real/predicted u, v, p)
points = real_data_closest_time[['x', 'y']].values
values_u_real = real_data_closest_time['Ux'].values
values_v_real = real_data_closest_time['Uy'].values
values_p_real = real_data_closest_time['p'].values
values_water_real = real_data_closest_time['alpha.water'].values

# Convert predicted tensors to numpy arrays for interpolation
values_u_pred = u_pred_closest_time.cpu().numpy().flatten()
values_v_pred = v_pred_closest_time.cpu().numpy().flatten()
values_p_pred = p_pred_closest_time.cpu().numpy().flatten()
values_water_pred = water_pred_closest_time.cpu().numpy().flatten()

# Interpolate the real and predicted data onto the grid
U_real_grid = griddata(points, values_u_real, (X_grid, Y_grid), method='cubic')
V_real_grid = griddata(points, values_v_real, (X_grid, Y_grid), method='cubic')
P_real_grid = griddata(points, values_p_real, (X_grid, Y_grid), method='cubic')
Water_real_grid = griddata(points, values_water_real, (X_grid, Y_grid), method='cubic')

U_pred_grid = griddata(points, values_u_pred, (X_grid, Y_grid), method='cubic')
V_pred_grid = griddata(points, values_v_pred, (X_grid, Y_grid), method='cubic')
P_pred_grid = griddata(points, values_p_pred, (X_grid, Y_grid), method='cubic')
Water_pred_grid = griddata(points, values_water_pred, (X_grid, Y_grid), method='cubic')

# Handle potential NaN values (fill with 0)
U_real_grid = np.nan_to_num(U_real_grid, nan=0.0)
V_real_grid = np.nan_to_num(V_real_grid, nan=0.0)
P_real_grid = np.nan_to_num(P_real_grid, nan=0.0)
Water_real_grid = np.nan_to_num(Water_real_grid, nan=0.0)

U_pred_grid = np.nan_to_num(U_pred_grid, nan=0.0)
V_pred_grid = np.nan_to_num(V_pred_grid, nan=0.0)
P_pred_grid = np.nan_to_num(P_pred_grid, nan=0.0)
Water_pred_grid = np.nan_to_num(Water_pred_grid, nan=0.0)

# Display the shapes of the interpolated grids
print("Shape of U_real_grid:", U_real_grid.shape)
print("Shape of V_real_grid:", V_real_grid.shape)
print("Shape of P_real_grid:", P_real_grid.shape)
print("Shape of U_pred_grid:", U_pred_grid.shape)
print("Shape of V_pred_grid:", V_pred_grid.shape)
print("Shape of P_pred_grid:", P_pred_grid.shape)

# Create a figure and a set of subplots
fig, ax = plt.subplots(4, 3, figsize=(15, 12))

# Define the extent for the heatmaps
extent = [x_unique.min(), x_unique.max(), y_unique.min(), y_unique.max()]

# Plot Real and Predicted Ux
im_u_real = ax[0, 0].imshow(U_real_grid, extent=extent, origin='lower', aspect='auto', cmap='viridis')
fig.colorbar(im_u_real, ax=ax[0, 0])
ax[0, 0].set_title(f'Real Ux Field at t={closest_time:.4f}')

im_u_pred = ax[0, 1].imshow(U_pred_grid, extent=extent, origin='lower', aspect='auto', cmap='viridis')
fig.colorbar(im_u_pred, ax=ax[0, 1])
ax[0, 1].set_title(f'Predicted Ux Field at t={closest_time:.4f}')

im_u_diff = ax[0, 2].imshow(np.abs(U_real_grid - U_pred_grid), extent=extent, origin='lower', aspect='auto', cmap='viridis')
fig.colorbar(im_u_diff, ax=ax[0, 2])
ax[0, 2].set_title(f'Absolute Error in Ux Field at t={closest_time:.4f}')

# Plot Real and Predicted Uy
im_v_real = ax[1, 0].imshow(V_real_grid, extent=extent, origin='lower', aspect='auto', cmap='viridis')
fig.colorbar(im_v_real, ax=ax[1, 0])
ax[1, 0].set_title(f'Real Uy Field at t={closest_time:.4f}')

im_v_pred = ax[1, 1].imshow(V_pred_grid, extent=extent, origin='lower', aspect='auto', cmap='viridis')
fig.colorbar(im_v_pred, ax=ax[1, 1])
ax[1, 1].set_title(f'Predicted Uy Field at t={closest_time:.4f}')

im_v_diff = ax[1, 2].imshow(np.abs(V_real_grid - V_pred_grid), extent=extent, origin='lower', aspect='auto', cmap='viridis')
fig.colorbar(im_v_diff, ax=ax[1, 2])
ax[1, 2].set_title(f'Absolute Error in Uy Field at t={closest_time:.4f}')

# Plot Real and Predicted p
im_p_real = ax[2, 0].imshow(P_real_grid, extent=extent, origin='lower', aspect='auto', cmap='viridis')
fig.colorbar(im_p_real, ax=ax[2, 0])
ax[2, 0].set_title(f'Real p Field at t={closest_time:.4f}')

P_pred_grid_mayor_cero = np.where(P_pred_grid > 0, P_pred_grid, 0)

im_p_pred = ax[2, 1].imshow(P_pred_grid_mayor_cero, extent=extent, origin='lower', aspect='auto', cmap='viridis')
fig.colorbar(im_p_pred, ax=ax[2, 1])
ax[2, 1].set_title(f'Predicted p Field at t={closest_time:.4f}')

im_p_diff = ax[2, 2].imshow(np.abs(P_real_grid - P_pred_grid_mayor_cero), extent=extent, origin='lower', aspect='auto', cmap='viridis')
fig.colorbar(im_p_diff, ax=ax[2, 2])
ax[2, 2].set_title(f'Absolute Error in p Field at t={closest_time:.4f}')

im_water_real = ax[3, 0].imshow(Water_real_grid, extent=extent, origin='lower', aspect='auto', cmap='viridis')
fig.colorbar(im_water_real, ax=ax[3, 0])
ax[3, 0].set_title(f'Real Water Field at t={closest_time:.4f}')

im_water_pred = ax[3, 1].imshow(Water_pred_grid, extent=extent, origin='lower', aspect='auto', cmap='viridis')
fig.colorbar(im_water_pred, ax=ax[3, 1])
ax[3, 1].set_title(f'Predicted Water Field at t={closest_time:.4f}')

im_water_diff = ax[3, 2].imshow(np.abs(Water_real_grid - Water_pred_grid), extent=extent, origin='lower', aspect='auto', cmap='viridis')
fig.colorbar(im_water_diff, ax=ax[3, 2])
ax[3, 2].set_title(f'Absolute Error in Water Field at t={closest_time:.4f}')

# Add a main title to the figure
fig.suptitle('Real vs Predicted Fields at Closest Time to t=1', fontsize=16)

# Adjust layout
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust rect to make space for suptitle

# Display the plot
plt.show()

NameError: name 'train_config' is not defined

In [ ]:
##CREAR GIF

import shutil
import imageio.v2 as imageio

x_bounds = [0.0, 0.584 / L_ref]
y_bounds = [0.0, 0.584 / L_ref]
t_bounds = [0.0, 5.0 / T_ref] # Rango de tiempo para el GIF
model_path = '/kaggle/input/mlp-dambreak/pytorch/default/1/NS_model_train (1).pt' # Assuming the last saved model is at step 5000

# Incrementos (como pediste)
dx = 0.05
dy = 0.05
dt = 0.05

# Carpeta temporal para guardar los frames
frame_dir = "./gif_frames"
# if os.path.exists(frame_dir):
#     shutil.rmtree(frame_dir) # Borra la carpeta si ya existe
# os.makedirs(frame_dir)

# print("Creando vectores de la grilla...")
# # Crear vectores de coordenadas (FÍSICAS)
# x_vec = np.arange(x_bounds[0], x_bounds[1] + dx, dx)
# y_vec = np.arange(y_bounds[0], y_bounds[1] + dy, dy)
# t_vec = np.arange(t_bounds[0], t_bounds[1] + dt, dt)

# # Crear malla 2D (xx, yy)
# xx, yy = np.meshgrid(x_vec, y_vec)

# # Aplanar la malla para la entrada de la red
# x_in_flat = xx.flatten().reshape(-1, 1)
# y_in_flat = yy.flatten().reshape(-1, 1)

# # Convertir a tensores (solo una vez)
# x_in_torch = torch.tensor(x_in_flat, dtype=torch.float32).to(device)
# y_in_torch = torch.tensor(y_in_flat, dtype=torch.float32).to(device)

# dummy_mean = np.zeros((1, train_config['dimension']))
# dummy_std = np.ones((1, train_config['dimension']))
# pinn_net = PINN_Net(layer_mat, dummy_mean, dummy_std, device, act=train_config['act'], norm=train_config['inner_Norm'])
# pinn_net = pinn_net.to(device)
# pinn_net.load_state_dict(torch.load(model_path, map_location=device))
# pinn_net.eval() # Poner el modelo en modo evaluación

# filenames = []
# print(f"Generando {len(t_vec)} frames...")

# for i, t_val in enumerate(t_vec):
#     # Crear el tensor de tiempo para este frame
#     # (todos los puntos de la malla tienen el mismo tiempo t_val)
#     t_in_torch = torch.full_like(x_in_torch, t_val)

#     # --- Predicción de la Red ---
#     with torch.no_grad():
#         # (x, y, t) son FÍSICOS. La red los normaliza internamente.
#         _, _, _, _, _, _, water_pred = pinn_net.forward(x_in_torch, y_in_torch, t_in_torch)

#     # Mover a CPU/numpy y reformar a la grilla 2D
#     water_grid = water_pred.cpu().numpy().reshape(yy.shape)

#     # --- Ploteo del Frame ---
#     plt.figure(figsize=(10, 5))
#     # Usar pcolormesh para un mapa de calor
#     plt.pcolormesh(xx, yy, water_grid, cmap='coolwarm', vmin=0.0, vmax=1.0, shading='gouraud')

#     plt.colorbar(label='alpha.water (0=Aire, 1=Agua)')
#     plt.xlabel("Posición X (m)")
#     plt.ylabel("Posición Y (m)")
#     plt.title(f"Predicción de Fase (alpha.water) | Tiempo t = {t_val:.3f} s")
#     plt.axis('scaled')
#     plt.clim(0, 1) # Forzar la barra de color a estar entre 0 y 1

#     # Guardar el frame
#     filename = f"{frame_dir}/frame_{i:04d}.png"
#     plt.savefig(filename)
#     plt.close()
#     filenames.append(filename)

#     if (i % 10 == 0):
#         print(f"  Frame {i}/{len(t_vec)} completado (t={t_val:.3f}s)")

# --- Creación del GIF ---
print("\nCompilando el GIF...")
output_gif = 'alpha_water_animation.gif'
with imageio.get_writer(output_gif, mode='I', duration=0.1) as writer:
    for filename in filenames:
        image = imageio.imread(filename)
        writer.append_data(image)

print(f"¡Animación guardada como '{output_gif}'!")

# --- Limpieza ---
print(f"Borrando la carpeta temporal '{frame_dir}'...")
shutil.rmtree(frame_dir)
print("Hecho.")


Compilando el GIF...
¡Animación guardada como 'alpha_water_animation.gif'!
Borrando la carpeta temporal './gif_frames'...
Hecho.
